# 🎬 EscenIA · Generador de imágenes (versión fácil, 1 botón)

**Solo 2 pasos:**
1. Menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4) → Guardar**.
2. Pulsa el botón **▶** de la única celda de abajo. (Si sale "este cuaderno no lo creó Google", dale **Ejecutar de todos modos**.)

Luego espera: instala todo, prepara el modelo y crea las imágenes de tu historia. Al final te descarga un **.zip**. La primera vez tarda varios minutos (descarga el modelo). ¡Eso es todo!

> Para cambiar la historia o el personaje, edita las líneas `historia = "..."` y `personaje = "..."` dentro de la celda.


In [ ]:
#@title ▶ Pulsa aquí para crear tu video (imágenes) { display-mode: "form" }

# ====== ESCRIBE TU HISTORIA AQUÍ ======
historia  = "Había una vez un pequeño dinosaurio llamado Rex. Vivía en un valle lleno de flores brillantes y cascadas de colores. Un día, Rex descubrió una cueva misteriosa escondida entre las rocas. Dentro brillaba un cristal mágico con todos los colores del arcoíris. Rex supo que esa noche comenzaría la aventura más grande de su vida."
personaje = "un pequeño dinosaurio verde de ojos grandes y amigables"
estilo    = "Animación 3D (Pixar)"   #@param ["Animación 3D (Pixar)", "Anime / Ghibli", "Acuarela infantil", "Realista cine", "Cómic"]
formato   = "Vertical (Shorts/TikTok)"  #@param ["Vertical (Shorts/TikTok)", "Horizontal (YouTube)", "Cuadrado"]
# ======================================

import subprocess, sys
print("⏳ Preparando el programa (1-2 min)...")
subprocess.run([sys.executable,"-m","pip","-q","install","diffusers","transformers","accelerate","safetensors","--upgrade"])

import torch, re, os, zipfile
from diffusers import StableDiffusionXLPipeline
from IPython.display import display

if not torch.cuda.is_available():
    raise SystemExit("❌ No hay GPU. Ve a Entorno de ejecución → Cambiar tipo → GPU (T4) → Guardar, y vuelve a pulsar ▶.")
print("✅ GPU:", torch.cuda.get_device_name(0))

print("⏳ Cargando el modelo (la primera vez descarga ~7 GB)...")
pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16, variant="fp16", use_safetensors=True
)
pipe.enable_model_cpu_offload()   # usa poca memoria de GPU
for _m in ("enable_attention_slicing","enable_vae_slicing","enable_vae_tiling"):
    try: getattr(pipe,_m)()
    except Exception: pass
print("✅ Modelo listo")

ESTILOS = {
 "Animación 3D (Pixar)":"Pixar-style 3D animated movie still, cinematic lighting, highly detailed, vibrant colors, soft global illumination, depth of field, charming character design",
 "Anime / Ghibli":"anime style, Studio Ghibli inspired, soft painterly background, cel shading, warm cinematic lighting",
 "Acuarela infantil":"children book watercolor illustration, soft pastel colors, hand painted, storybook style",
 "Realista cine":"cinematic photography, ultra realistic, dramatic lighting, 8k, film still",
 "Cómic":"comic book illustration, bold ink lines, dynamic shading, graphic novel style",
}
DIMS = {"Vertical (Shorts/TikTok)":(768,1024),"Horizontal (YouTube)":(1024,768),"Cuadrado":(896,896)}
NEG = "blurry, low quality, deformed, extra limbs, text, watermark, ugly, bad anatomy"

def trocear(t):
    parts=re.split(r"(?<=[\.\!\?…])\s+", re.sub(r"\s+"," ",t).strip())
    out=[]; buf=""
    for p in parts:
        p=p.strip()
        if not p: continue
        if len((buf+" "+p).strip())<45: buf=(buf+" "+p).strip()
        else:
            if buf: out.append(buf)
            buf=p
    if buf: out.append(buf)
    return out[:12]

frases=trocear(historia); w,h=DIMS[formato]; est=ESTILOS[estilo]
os.makedirs("/content/escenas",exist_ok=True)
for f in os.listdir("/content/escenas"): os.remove("/content/escenas/"+f)
print(f"\n🎬 Creando {len(frases)} escenas...\n")
gen=torch.Generator("cuda"); rutas=[]
for i,fr in enumerate(frases):
    gen.manual_seed(100+i*7)
    img=pipe(prompt=f"{fr}. Featuring {personaje}. {est}", negative_prompt=NEG,
             width=w,height=h,num_inference_steps=30,guidance_scale=7.0,generator=gen).images[0]
    r=f"/content/escenas/escena_{i+1:02d}.png"; img.save(r); rutas.append(r)
    print(f"  ✓ Escena {i+1}: {fr[:55]}...")
    display(img.resize((w//3,h//3)))
    import gc; del img; gc.collect(); torch.cuda.empty_cache()

zp="/content/mis_escenas.zip"
with zipfile.ZipFile(zp,"w") as z:
    for r in rutas: z.write(r,os.path.basename(r))
print("\n✅ ¡Listo! Descargando mis_escenas.zip ...")
from google.colab import files; files.download(zp)


## 🎬 Celda 2 · Armar el VIDEO final (voz + subtítulos + música)
Ejecuta esta celda **después** de generar las imágenes. Toma las escenas, les pone narración en español, subtítulos, movimiento de cámara y música suave, y te descarga el **video.mp4** listo para tu canal.


In [ ]:
#@title ▶ Pulsa aquí para crear el VIDEO final { display-mode: "form" }
import subprocess, sys, os, glob, math, re, textwrap
print("⏳ Instalando lo necesario para el video...")
subprocess.run([sys.executable,"-m","pip","-q","install","gTTS"])
subprocess.run(["apt-get","-qq","install","-y","ffmpeg"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
from gtts import gTTS
from PIL import Image, ImageDraw, ImageFont
import numpy as np

# --- recupera la historia y las escenas ---
try:
    historia
except NameError:
    historia = "Había una vez un pequeño dinosaurio llamado Rex. Vivía en un valle lleno de flores. Un día descubrió una cueva misteriosa. Dentro brillaba un cristal mágico. Esa noche comenzó su gran aventura."
def trocear(t):
    parts=re.split(r"(?<=[\.\!\?…])\s+", re.sub(r"\s+"," ",t).strip())
    out=[];buf=""
    for p in parts:
        p=p.strip()
        if not p: continue
        if len((buf+" "+p).strip())<45: buf=(buf+" "+p).strip()
        else:
            if buf: out.append(buf)
            buf=p
    if buf: out.append(buf)
    return out[:12]
frases=trocear(historia)
imgs=sorted(glob.glob("/content/escenas/*.png"))
n=min(len(frases),len(imgs))
frases=frases[:n]; imgs=imgs[:n]
assert n>0, "No hay imágenes. Ejecuta primero la celda de generar imágenes."
print(f"\U0001F3AC Montando video de {n} escenas...")

FPS=24
W,H = Image.open(imgs[0]).size
os.makedirs("/content/frames", exist_ok=True)
for f in glob.glob("/content/frames/*.png"): os.remove(f)
os.makedirs("/content/aud", exist_ok=True)
for f in glob.glob("/content/aud/*"): os.remove(f)

# fuente
try:
    FP="/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
    font=ImageFont.truetype(FP, int(W*0.052))
except Exception:
    font=ImageFont.load_default()

def ffdur(path):
    out=subprocess.run(["ffprobe","-v","quiet","-show_entries","format=duration","-of","csv=p=0",path],capture_output=True,text=True).stdout.strip()
    try: return float(out)
    except: return 3.0

def draw_sub(im, texto):
    d=ImageDraw.Draw(im,"RGBA")
    # viñeta inferior
    d.rectangle([0,int(H*0.74),W,H], fill=(0,0,0,110))
    maxw=int(W*0.86); lines=[]; line=""
    for word in texto.split():
        t=(line+" "+word).strip()
        if d.textlength(t,font=font)>maxw and line: lines.append(line); line=word
        else: line=t
    if line: lines.append(line)
    lh=int(W*0.066); y0=int(H*0.9)-(len(lines)-1)*lh
    for i,ln in enumerate(lines):
        w=d.textlength(ln,font=font); x=(W-w)/2; y=y0+i*lh
        for dx,dy in [(-2,0),(2,0),(0,-2),(0,2),(-2,-2),(2,2)]:
            d.text((x+dx,y+dy),ln,font=font,fill=(0,0,0,255))
        d.text((x,y),ln,font=font,fill=(255,255,255,255))
    return im

fi=0
for i,(img,txt) in enumerate(zip(imgs,frases)):
    # narración
    mp3=f"/content/aud/s{i:02d}.mp3"
    gTTS(txt, lang="es").save(mp3)
    dur=max(2.4, ffdur(mp3)+0.5)
    nf=int(dur*FPS)
    base=Image.open(img).convert("RGB")
    for k in range(nf):
        t=k/max(1,nf-1)
        z=1.06+0.12*t
        nw,nh=int(W*z),int(H*z)
        fr=base.resize((nw,nh), Image.LANCZOS)
        dx=int((nw-W)*(0.5+(t-0.5)*0.4)); dy=int((nh-H)*(0.5+(t-0.5)*0.2))
        fr=fr.crop((dx,dy,dx+W,dy+H))
        fr=draw_sub(fr, txt)
        fr.save(f"/content/frames/f_{fi:05d}.png"); fi+=1
    print(f"  ✓ Escena {i+1}: {txt[:45]}...")

# une la narración en un solo audio
listtxt="|".join(f"/content/aud/s{i:02d}.mp3" for i in range(n))
subprocess.run(f'ffmpeg -y -i "concat:{listtxt}" -acodec libmp3lame /content/narracion.mp3',shell=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
total=ffdur("/content/narracion.mp3")

# música suave (pad de acordes con numpy)
sr=44100; tt=np.linspace(0,total,int(sr*total),endpoint=False)
notes=[220.0,277.18,329.63]; music=np.zeros_like(tt)
for i,f in enumerate(notes):
    music+=0.33*np.sin(2*np.pi*f*tt)*(0.6+0.4*np.sin(2*np.pi*0.05*tt+i))
music*= (0.08/ (np.max(np.abs(music))+1e-6))
import wave
with wave.open("/content/music.wav","w") as wv:
    wv.setnchannels(1); wv.setsampwidth(2); wv.setframerate(sr)
    wv.writeframes((music*32767).astype(np.int16).tobytes())

print("⏳ Renderizando el video (puede tardar 1-2 min)...")
cmd=('ffmpeg -y -framerate 24 -i /content/frames/f_%05d.png '
     '-i /content/narracion.mp3 -i /content/music.wav '
     '-filter_complex "[2:a]volume=0.5[m];[1:a][m]amix=inputs=2:duration=first:dropout_transition=0[a]" '
     '-map 0:v -map "[a]" -c:v libx264 -pix_fmt yuv420p -r 24 -c:a aac -shortest /content/video.mp4')
r=subprocess.run(cmd,shell=True,capture_output=True,text=True)
if not os.path.exists("/content/video.mp4"):
    # respaldo sin música
    subprocess.run('ffmpeg -y -framerate 24 -i /content/frames/f_%05d.png -i /content/narracion.mp3 -map 0:v -map 1:a -c:v libx264 -pix_fmt yuv420p -r 24 -c:a aac -shortest /content/video.mp4',shell=True)

print("✅ ¡Video listo! Mostrando y descargando...")
from IPython.display import HTML
import base64
with open("/content/video.mp4","rb") as f: b64=base64.b64encode(f.read()).decode()
display(HTML(f'<video width="320" controls autoplay loop src="data:video/mp4;base64,{b64}"></video>'))
from google.colab import files
files.download("/content/video.mp4")
